# FGSM Vision Engine - Hitbox Segmenter (Kaggle)

Continues training from `checkpoint-3200` (the same run started on Colab).

**Before running:**
1. Notebook Settings (right sidebar) -> Accelerator -> GPU T4 x2 (or P100).
2. Notebook Settings -> Internet -> On.
3. Add Data -> attach your `fgsm-hitbox-checkpoint` dataset (see step below
   for how that dataset gets created/updated).

**First time only** - getting checkpoint-3200 in:
1. Download the checkpoint-3200 folder from Google Drive as a zip
   (right-click the folder in Drive -> Download).
2. On kaggle.com -> Datasets -> New Dataset -> upload that zip, name it
   `fgsm-hitbox-checkpoint`, keep it Private.
3. Attach it to this notebook via "Add Data" in the sidebar.

**Every session after that** - when you're done for the session (approaching
the 12hr limit, or just pausing), click "Save Version" (Save & Run All).
That commits everything under `/kaggle/working/` - including whatever new
checkpoints got written - as this notebook's own output. Next session,
attach *this notebook's output* (not the original zip) as the data source
instead, and it picks up from the latest checkpoint automatically.

In [ ]:
!git clone https://github.com/lionelishi21/fgsm-vision-engine.git
%cd fgsm-vision-engine

In [ ]:
!pip install -r requirements.txt -q

In [ ]:
# Paste a freshly-generated presigned URL here at runtime - never hardcode
# one in this notebook. Ask Claude for a fresh one (they expire in 1hr):
#   aws s3 presign s3://fgsm-vision-models-aibridix-official/fgsm_colab_package.zip --expires-in 3600 --profile aibridix_official
from getpass import getpass
package_url = getpass("Presigned S3 URL for fgsm_colab_package.zip: ")
!wget -q -O /kaggle/working/fgsm_colab_package.zip "{package_url}"
!unzip -o -q /kaggle/working/fgsm_colab_package.zip -d /kaggle/working/
!cp -r /kaggle/working/fgsm_colab_package/data .

In [ ]:
import os, glob, json, shutil

OUTPUT_DIR = "/kaggle/working/fgsm_runs/hitbox_segmenter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Find whatever checkpoint dataset got attached under /kaggle/input - either
# the original one-time zip upload (checkpoint-3200/...) or a previous
# session's own committed output (fgsm_runs/hitbox_segmenter/checkpoint-N/...)
# - and copy its contents into the writable output dir.
candidates = glob.glob("/kaggle/input/*/checkpoint-*") + \
             glob.glob("/kaggle/input/*/fgsm_runs/hitbox_segmenter/checkpoint-*")
if not candidates:
    raise RuntimeError(
        "No checkpoint found under /kaggle/input - did you attach the "
        "fgsm-hitbox-checkpoint dataset (or a previous session's output) "
        "via Add Data?"
    )
latest = max(candidates, key=lambda p: int(p.rstrip("/").split("-")[-1]))
dest = os.path.join(OUTPUT_DIR, os.path.basename(latest))
if not os.path.exists(dest):
    shutil.copytree(latest, dest)
print(f"Seeded {dest}")

# The full training run's expected training-split size (from
# annotations.json, 80% split) - train_hitbox_segmentation.py refuses to
# resume from a checkpoint unless this matches exactly, to guard against
# silently continuing training on a different dataset. Colab's checkpoints
# never had this file, so write it if it's missing here too.
fingerprint_path = os.path.join(OUTPUT_DIR, "dataset_fingerprint.json")
if not os.path.exists(fingerprint_path):
    with open(fingerprint_path, "w") as f:
        json.dump({"num_train_images": 3486}, f)
    print(f"Wrote {fingerprint_path}")

In [ ]:
!python src/train_hitbox_segmentation.py --data data/ufd/segmentation_dataset --output /kaggle/working/fgsm_runs/hitbox_segmenter

When you're pausing this session (12hr limit approaching, or just done for
now), click **Save Version** in the top right. That commits
`/kaggle/working/fgsm_runs/hitbox_segmenter` as this notebook's output.

Next session: open a new session of this notebook, remove the old data
source, and **Add Data -> Notebook Output Files -> this notebook** instead.
The seeding cell above will pick up the latest checkpoint automatically.